# Les 02 - Verkennen van het Microsoft Agent Framework

Het **Microsoft Agent Framework (MAF)** is een uniform framework voor het bouwen van AI-agents. Het biedt een schone, samenstelbare architectuur met vier kernonderdelen:

- **Client** – maakt verbinding met een AI-model endpoint en verzorgt de communicatie
- **Agent** – wikkelt een client in met instructies en definitie van tools
- **Tools** – breiden de mogelijkheden van de agent uit met aangepaste functies die het model kan aanroepen
- **Session** – houdt de gesprekshistorie bij voor interacties met meerdere beurten

In deze les bouwen we een **reisboekingsagent** die de beschikbaarheid van bestemmingen controleert met behulp van deze concepten.


## Installatie


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Inzicht in de Architectuur van het Agent Framework

Het Microsoft Agent Framework volgt een gelaagde architectuur:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Een `FoundryChatClient` maakt verbinding met een Azure OpenAI-implementatie. Het handelt authenticatie, aanvraagformatering en antwoordparsing af.
2. **Agent** – Gemaakt vanuit de client via `provider.create_agent()`, combineert de agent modeltoegang met instructies (systeemprompt) en tools.
3. **Tools** – Python-functies gedecoreerd met `@tool` die de agent kan aanroepen om acties uit te voeren of gegevens op te halen.
4. **Sessie** – Een `AgentSession`-object (gemaakt via `agent.create_session()`) dat het gesprek opslaat, waardoor meer beurten dialoog mogelijk is waarbij de agent eerdere context onthoudt.

Laten we elke laag stap voor stap opbouwen.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Tools Toevoegen met de @tool Decorator

Tools laten agents handelingen uitvoeren die verder gaan dan het genereren van tekst. De `@tool` decorator zet een gewone Python-functie om in iets dat de agent kan aanroepen.

Belangrijke punten:
- Gebruik `Annotated[type, "description"]` zodat het model elk parameter begrijpt.
- De docstring wordt de toolbeschrijving die het model ziet.
- `approval_mode="never_require"` betekent dat de tool automatisch wordt uitgevoerd zonder gebruikersbevestiging.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Een agent maken met hulpmiddelen

Nu combineren we de client, instructies en hulpmiddelen in een agent. De `instructions` fungeren als het systeemprompt — ze definiëren de persoonlijkheid en het gedrag van de agent.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Meerbeurtgesprekken met Sessies

Een `AgentSession` (gemaakt via `agent.create_session()`) houdt alle berichten in een gesprek bij. Door dezelfde sessie mee te geven aan elke `agent.run()`-aanroep, heeft de agent toegang tot de volledige gespreksgeschiedenis en kan hij terugverwijzen naar eerdere berichten.

We geven `tools=[check_destination_availability]` mee zodat de agent tijdens elke beurt onze beschikbaarheidschecker kan aanroepen.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Samenvatting

In deze les heb je de vier pijlers van het Microsoft Agent Framework verkend:

| Concept | Wat je hebt geleerd |
|---------|------------------|
| **Client** | `FoundryChatClient` maakt verbinding met Azure OpenAI met op referenties gebaseerde authenticatie |
| **Agent** | `provider.create_agent()` bundelt een modelverbinding met instructies en een naam |
| **Tools** | De `@tool` decorator stelt Python-functies beschikbaar die de agent kan aanroepen |
| **Session** | `agent.create_session()` onderhoudt het gespreksverleden over meerdere beurten |

Deze bouwstenen vormen samen agents die natuurlijke gesprekken kunnen voeren, externe functies kunnen aanroepen en context kunnen behouden — de basis voor meer geavanceerde agentpatronen in latere lessen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
